# Банковский ИИ-ассистент: Prompting → RAG → Fine-Tuning (LoRA) → Гибрид

Практическая работа по адаптации LLM на примере ответов о тарифах и продуктах банка.

**Модель:** `Qwen/Qwen3-4B-Instruct-2507` (4-bit, T4 Colab)

**Репозиторий:** [llm-adaptation-banking-agent](https://github.com/pueraeternis/llm-adaptation-banking-agent)

## 0. Подготовка окружения (Colab)

В Colab выберите **Среда выполнения → Сменить среду выполнения → T4 GPU**.

Опционально: **Secrets** → добавьте `HF_TOKEN` ([токен Hugging Face](https://huggingface.co/settings/tokens)) — убирает предупреждения о лимитах загрузки.

После установки зависимостей включается **режим тихого лога** (`warnings` + `logging`) — скрывает безопасный шум (bitsandbytes, HF Hub, BertModel LOAD REPORT и т.п.), не скрывая реальные ошибки.

In [14]:
import os

if not os.path.exists("llm-adaptation-banking-agent"):
    print("Клонирование репозитория...")
    !git clone https://github.com/pueraeternis/llm-adaptation-banking-agent.git
    %cd llm-adaptation-banking-agent
else:
    print("Репозиторий уже на месте, переходим в каталог проекта.")
    %cd llm-adaptation-banking-agent

Клонирование репозитория...
Cloning into 'llm-adaptation-banking-agent'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 29 (delta 6), reused 23 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 18.74 KiB | 314.00 KiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/llm-adaptation-banking-agent/llm-adaptation-banking-agent


In [15]:
print("Установка зависимостей...")
%pip install -q -r requirements.txt

import logging
import os
import warnings


def setup_notebook_quiet() -> None:
    """Скрывает шумные, но безопасные предупреждения библиотек (Colab)."""
    os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
    os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
    os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

    warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes")
    warnings.filterwarnings("ignore", category=FutureWarning, module="torch")
    warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

    for logger_name in (
        "transformers",
        "transformers.modeling_utils",
        "sentence_transformers",
        "huggingface_hub",
        "huggingface_hub.utils._auth",
        "datasets",
    ):
        logging.getLogger(logger_name).setLevel(logging.ERROR)

    print("Режим тихого лога: включён (критические ошибки по-прежнему видны).")


setup_notebook_quiet()

Установка зависимостей...
Режим тихого лога: включён (критические ошибки по-прежнему видны).


In [16]:
def setup_huggingface() -> None:
    """Опциональная авторизация HF Hub (Colab Secrets: HF_TOKEN)."""
    import os

    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from google.colab import userdata

            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None

    if token:
        os.environ["HF_TOKEN"] = token
        from huggingface_hub import login

        login(token=token, add_to_git_credential=False)
        print("Hugging Face Hub: авторизация выполнена.")
    else:
        print(
            "Hugging Face Hub: без токена (публичные модели загружаются).\n"
            "  Чтобы убрать предупреждения: Colab → Secrets → HF_TOKEN"
        )


setup_huggingface()

Hugging Face Hub: авторизация выполнена.


In [17]:
from pathlib import Path

import numpy as np
import torch
import faiss
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig,
)
from trl import SFTConfig, SFTTrainer

ROOT = Path(".").resolve()
for candidate in [Path("."), Path(".."), Path("/content/llm-adaptation-banking-agent")]:
    if (candidate / "data" / "banking_tariffs.txt").exists():
        ROOT = candidate.resolve()
        break

DATA_DIR = ROOT / "data"
TARIFFS_PATH = DATA_DIR / "banking_tariffs.txt"
LORA_DATASET_PATH = DATA_DIR / "lora_dataset.jsonl"

assert TARIFFS_PATH.exists(), f"Не найден {TARIFFS_PATH}"
assert LORA_DATASET_PATH.exists(), f"Не найден {LORA_DATASET_PATH}"

print("Корень проекта:", ROOT)
print("База знаний RAG:", TARIFFS_PATH)
print("LoRA датасет:", LORA_DATASET_PATH)
print("GPU доступен:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Корень проекта: /content/llm-adaptation-banking-agent/llm-adaptation-banking-agent
База знаний RAG: /content/llm-adaptation-banking-agent/llm-adaptation-banking-agent/data/banking_tariffs.txt
LoRA датасет: /content/llm-adaptation-banking-agent/llm-adaptation-banking-agent/data/lora_dataset.jsonl
GPU доступен: True
GPU: Tesla T4


In [18]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Загрузка токенизатора: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Загрузка модели в 4-bit (это займёт 1–3 мин)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Модель и токенизатор готовы.")

Загрузка токенизатора: Qwen/Qwen3-4B-Instruct-2507...
Загрузка модели в 4-bit (это займёт 1–3 мин)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Модель и токенизатор готовы.


In [19]:
def generate_response(messages: list[dict], max_new_tokens: int = 256) -> str:
    """Генерация ответа chat-модели через apply_chat_template."""
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    print("Генерация ответа...")
    with torch.no_grad():
        output_ids = model.generate(**inputs, generation_config=gen_config)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 1. Prompt Engineering

Базовый ответ модели **без базы знаний**. Системный промпт запрещает выдумывать цифры — модель честно отказывается называть тариф (это ожидаемо). В разделе 2 RAG подставит факты из `banking_tariffs.txt`.

In [20]:
SYSTEM_PROMPT = """Ты — вежливый банковский ассистент.
Отвечай только на вопросы о продуктах и тарифах банка.
Если не знаешь ответа — честно скажи, что уточнишь у специалиста.
Не придумывай цифры и условия."""

USER_QUESTION = "Сколько стоит обслуживание дебетовой карты «Стандарт»?"

print("Вопрос пользователя:", USER_QUESTION)

print("\n--- A) Без системного промпта (модель может галлюцинировать) ---\n")
bare_messages = [{"role": "user", "content": USER_QUESTION}]
bare_answer = generate_response(bare_messages, max_new_tokens=200)
print(bare_answer)

prompt_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUESTION},
]

print("\n--- B) С системным промптом, без RAG (ожидаем отказ от выдуманных цифр) ---\n")
prompt_answer = generate_response(prompt_messages)
print(prompt_answer)

print(
    "\nВывод: без внешней базы знаний точный тариф недоступен → "
    "подключаем RAG в разделе 2."
)

Вопрос пользователя: Сколько стоит обслуживание дебетовой карты «Стандарт»?

--- A) Без системного промпта (модель может галлюцинировать) ---

Генерация ответа...
Обслуживание дебетовой карты «Стандарт» (например, дебетовой карты от Сбербанка, если речь идёт о конкретной марке) может варьироваться в зависимости от банка, типа карты и условий обслуживания.

На текущий момент (май 2024 года) **дебетовая карта «Стандарт» от Сбербанка**:

- **Бесплатно** (в рамках определённых условий).

### Подробности:
- Сбербанк предлагает дебетовые карты по цене **без комиссий** за обслуживание, если вы используете карту в рамках лимитов и не совершаете определённые операции (например, не выезжаете за пределы страны, не используете кредитные функции).
- **Стоимость обслуживания** (

--- B) С системным промптом, без RAG (ожидаем отказ от выдуманных цифр) ---

Генерация ответа...
Стоимость обслуживания дебетовой карты «Стандарт» уточню у специалиста.

Вывод: без внешней базы знаний точный тариф недоступе

## 2. RAG (Retrieval-Augmented Generation)

Векторный поиск по `banking_tariffs.txt` (FAISS + `sentence-transformers`), топ-1 чанк в контекст промпта.

In [21]:
EMBED_MODEL_ID = "paraphrase-multilingual-MiniLM-L12-v2"

print("Подготовка чанков из базы знаний...")
tariffs_text = TARIFFS_PATH.read_text(encoding="utf-8").strip()
raw_parts = tariffs_text.split("\n## ")
chunks = []
for part in raw_parts:
    part = part.strip()
    if not part:
        continue
    if not part.startswith("##"):
        part = "## " + part
    chunks.append(part)

print(f"Чанков в индексе: {len(chunks)}")
for i, ch in enumerate(chunks):
    title = ch.split("\n", 1)[0]
    print(f"  [{i}] {title}")

Подготовка чанков из базы знаний...
Чанков в индексе: 3
  [0] ## Дебетовая карта «Стандарт»
  [1] ## Вклад «Весенний 2026»
  [2] ## Кредитная карта «Платинум»


In [22]:
print(f"Загрузка embedding-модели: {EMBED_MODEL_ID}...")
embed_model = SentenceTransformer(EMBED_MODEL_ID)

print("Векторизация чанков и построение FAISS IndexFlatL2...")
chunk_embeddings = embed_model.encode(chunks, convert_to_numpy=True).astype("float32")
dimension = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(chunk_embeddings)
print(f"Индекс FAISS построен: {faiss_index.ntotal} векторов, размерность {dimension}.")


def retrieve_top1(query: str) -> tuple[str, float]:
    query_embedding = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = faiss_index.search(query_embedding, 1)
    idx = int(indices[0][0])
    return chunks[idx], float(distances[0][0])

Загрузка embedding-модели: paraphrase-multilingual-MiniLM-L12-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Векторизация чанков и построение FAISS IndexFlatL2...
Индекс FAISS построен: 3 векторов, размерность 384.


In [23]:
rag_query = USER_QUESTION
print("Запрос для RAG:", rag_query)

top_chunk, distance = retrieve_top1(rag_query)
print(f"\nНайден релевантный чанк (L2={distance:.4f}):\n{top_chunk}\n")

rag_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": (
            f"Контекст из базы знаний банка:\n{top_chunk}\n\n"
            f"Вопрос клиента: {rag_query}\n"
            "Ответь, опираясь только на контекст."
        ),
    },
]

print("--- Ответ модели (RAG) ---\n")
rag_answer = generate_response(rag_messages)
print(rag_answer)

Запрос для RAG: Сколько стоит обслуживание дебетовой карты «Стандарт»?

Найден релевантный чанк (L2=7.2301):
## Дебетовая карта «Стандарт»
Стоимость обслуживания дебетовой карты «Стандарт» составляет 0 рублей в месяц при совершении покупок на сумму от 10 000 рублей в месяц. В иных случаях стоимость обслуживания составляет 99 рублей в месяц. Снятие наличных в банкоматах банка бесплатное.

--- Ответ модели (RAG) ---

Генерация ответа...
Обслуживание дебетовой карты «Стандарт» составляет 0 рублей в месяц при совершении покупок на сумму от 10 000 рублей в месяц. В остальных случаях — 99 рублей в месяц.


## 3. Fine-Tuning (LoRA)

Дообучение адаптера на `lora_dataset.jsonl` (формат `messages`, JSON-ответы API-агента).

In [24]:
print("Загрузка датасета для LoRA...")
train_dataset = load_dataset(
    "json",
    data_files=str(LORA_DATASET_PATH),
    split="train",
)
print(f"Примеров в датасете: {len(train_dataset)}")
print("\nПример диалога:")
for msg in train_dataset[0]["messages"]:
    preview = msg["content"][:120] + ("..." if len(msg["content"]) > 120 else "")
    print(f"  [{msg['role']}]: {preview}")

Загрузка датасета для LoRA...


Generating train split: 0 examples [00:00, ? examples/s]

Примеров в датасете: 3

Пример диалога:
  [system]: Ты — банковский API-агент. Отвечай только в формате JSON с ключами 'response' и 'compliance_risk'.
  [user]: Могу ли я снять деньги с вклада Весенний раньше срока?
  [assistant]: {"response": "При досрочном расторжении вклада 'Весенний 2026' проценты выплачиваются по ставке вклада 'До востребования...


In [25]:
def formatting_func(example: dict) -> str:
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
    )


print("Подготовка модели к k-bit обучению и подключение LoRA...")
model.train()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Подготовка модели к k-bit обучению и подключение LoRA...
trainable params: 2,949,120 || all params: 4,025,417,216 || trainable%: 0.0733


In [26]:
sft_config = SFTConfig(
    output_dir="./lora_banking_agent",
    max_steps=15,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    logging_first_step=True,
    save_steps=15,
    report_to="none",
    bf16=torch.cuda.is_available(),
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)

print("Старт обучения LoRA (15 шагов)...")
train_result = trainer.train()

losses = [entry["loss"] for entry in trainer.state.log_history if "loss" in entry]
if losses:
    print(f"Обучение завершено. Loss (последний шаг): {losses[-1]:.4f}")
    print(f"Loss (средний за прогон): {sum(losses) / len(losses):.4f}")
elif hasattr(train_result, "training_loss") and train_result.training_loss is not None:
    print(f"Обучение завершено. Loss: {train_result.training_loss:.4f}")
else:
    print("Обучение завершено.")

model.eval()

Applying formatting function to train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Старт обучения LoRA (15 шагов)...
{'loss': '2.829', 'grad_norm': '2.812', 'learning_rate': '0.0002', 'entropy': '0.8475', 'num_tokens': '363', 'mean_token_accuracy': '0.6094', 'epoch': '1'}
{'loss': '2.649', 'grad_norm': '1.586', 'learning_rate': '0.0001867', 'entropy': '0.8625', 'num_tokens': '726', 'mean_token_accuracy': '0.634', 'epoch': '2'}
{'loss': '2.52', 'grad_norm': '1.5', 'learning_rate': '0.0001733', 'entropy': '0.8763', 'num_tokens': '1089', 'mean_token_accuracy': '0.6321', 'epoch': '3'}
{'loss': '2.384', 'grad_norm': '1.562', 'learning_rate': '0.00016', 'entropy': '0.9028', 'num_tokens': '1452', 'mean_token_accuracy': '0.6438', 'epoch': '4'}
{'loss': '2.251', 'grad_norm': '1.656', 'learning_rate': '0.0001467', 'entropy': '0.9345', 'num_tokens': '1815', 'mean_token_accuracy': '0.641', 'epoch': '5'}
{'loss': '2.116', 'grad_norm': '1.695', 'learning_rate': '0.0001333', 'entropy': '0.9721', 'num_tokens': '2178', 'mean_token_accuracy': '0.641', 'epoch': '6'}
{'loss': '1.997', '

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linea

## 4. Гибрид (LoRA + RAG)

RAG находит актуальный тариф, дообученная LoRA-модель отвечает в JSON-формате API-агента.

In [27]:
API_SYSTEM_PROMPT = (
    "Ты — банковский API-агент. Отвечай только одним JSON-объектом с ключами "
    "'response' и 'compliance_risk'. "
    "Поле response — развёрнутое предложение с цифрами и обоснованием строго по контексту. "
    "Поле compliance_risk — одно из: low, medium, high; "
    "ставь medium, если клиент не выполняет условие бесплатного обслуживания (порог 10 000 руб./мес)."
)

HYBRID_QUESTION = (
    "Сколько стоит обслуживание карты Стандарт если я трачу 5000 рублей?"
)

EXPECTED_HYBRID_JSON = (
    '{"response": "Поскольку сумма покупок менее 10 000 рублей в месяц, '
    'стоимость обслуживания карты \'Стандарт\' составит 99 рублей в месяц.", '
    '"compliance_risk": "medium"}'
)

print("Гибридный запрос:", HYBRID_QUESTION)
print("\nШаг 1: поиск контекста через FAISS (RAG)...")
hybrid_chunk, hybrid_distance = retrieve_top1(HYBRID_QUESTION)
print(f"Найден чанк (L2={hybrid_distance:.4f}):\n{hybrid_chunk}\n")

hybrid_messages = [
    {"role": "system", "content": API_SYSTEM_PROMPT},
    {
        "role": "user",
        "content": (
            f"Контекст из базы знаний банка:\n{hybrid_chunk}\n\n"
            f"Вопрос клиента: {HYBRID_QUESTION}\n\n"
            "Сформируй JSON-ответ, используя только факты из контекста."
        ),
    },
]

print("Шаг 2: генерация JSON через дообученную LoRA-модель...")
hybrid_answer = generate_response(hybrid_messages, max_new_tokens=512)

print("\n--- Финальный ответ (LoRA + RAG) ---\n")
print(hybrid_answer)

print("\n--- Эталон из обучающего датасета (для сравнения) ---\n")
print(EXPECTED_HYBRID_JSON)

Гибридный запрос: Сколько стоит обслуживание карты Стандарт если я трачу 5000 рублей?

Шаг 1: поиск контекста через FAISS (RAG)...
Найден чанк (L2=8.3955):
## Дебетовая карта «Стандарт»
Стоимость обслуживания дебетовой карты «Стандарт» составляет 0 рублей в месяц при совершении покупок на сумму от 10 000 рублей в месяц. В иных случаях стоимость обслуживания составляет 99 рублей в месяц. Снятие наличных в банкоматах банка бесплатное.

Шаг 2: генерация JSON через дообученную LoRA-модель...
Генерация ответа...

--- Финальный ответ (LoRA + RAG) ---

{"response": "Стоимость обслуживания дебетовой карты «Стандарт» составляет 99 рублей в месяц, так как сумма трат составляет 5000 рублей, что меньше порога в 10 000 рублей в месяц.", "compliance_risk": "medium"}

--- Эталон из обучающего датасета (для сравнения) ---

{"response": "Поскольку сумма покупок менее 10 000 рублей в месяц, стоимость обслуживания карты 'Стандарт' составит 99 рублей в месяц.", "compliance_risk": "medium"}


## 5. Сравнение методов

| Метод | Плюсы | Минусы |
|-------|-------|--------|
| Prompt Engineering | Быстро, без обучения | Модель может галлюцинировать без фактов |
| RAG | Актуальные знания без переобучения | Зависит от качества поиска и базы |
| LoRA | Фиксированный JSON-формат ответа | Нужны данные и GPU; факты устаревают |
| **LoRA + RAG** | Точные факты + нужный формат | Сложнее пайплайн, два компонента |